In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 9.1 Qubits, Gates, and the Bloch Sphere

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Chapter IX — Quantum Information as Linear Algebra",
    number="9.1",
    title="Qubits, Gates, and the Bloch Sphere",
    blurb="A qubit is a unit vector in C^2, a gate is a 2x2 unitary, and the "
    "whole apparatus lives on a sphere read off three Pauli expectation "
    "values. The strangest fact in the subject is a linear-algebra fact: "
    "rotating a qubit by a full turn multiplies it by minus one, and an "
    "interference experiment can read the sign.",
    difficulty="advanced",
    estimate="105–135 min",
)

## Notebook overview

[§3.4](../03-eigenvalues/hermitian-unitary-normal.ipynb) built the complex
inner product, the Pauli matrices, and a two-level state's **Bloch
vector**. This notebook takes those parts and assembles the smallest
quantum system — not as physics folklore but as linear algebra with every
claim gated: states are unit vectors in $\mathbb{C}^2$ modulo a global
phase, gates are the unitary group acting on them, and the sphere is a
faithful picture because three quadratic forms determine a state exactly
up to that phase.

The centrepiece is the rotation family $R_{\mathbf{n}}(\theta) =
\exp(-i\theta\,\mathbf{n}\cdot\boldsymbol{\sigma}/2)$, built two ways that
share no code — [§3.6](../03-eigenvalues/matrix-functions-exponential.ipynb)'s
spectral exponential and a closed form the reader derives — and checked
against each other at rounding. Its strangest property is gated rather
than narrated: the **double cover**. A $2\pi$ rotation of the Bloch vector
returns every *observable* to its start, yet multiplies the state by $-1$
— and an interference term $\langle\psi|R(\theta)|\psi\rangle$ tracks the
sign in plain sight, crossing $-1$ exactly at $\theta = 2\pi$.

The chapter's exact-versus-float thread starts here too: the $T$ gate's
eighth power is the identity — *exactly*, in SymPy, because
$e^{i\pi/4}$ is an eighth root of unity — while the float route drifts by
a measured few $\varepsilon$. Both facts are true; the course has spent
nine chapters learning to say which kind of true.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Nielsen and Chuang {cite}`nielsen2010` Chapters 1–2 and 4 are
> the standard treatment; the su(2) machinery is any quantum text's, e.g.
> Sakurai's opening chapters, and everything used here was built in
> [§3.4](../03-eigenvalues/hermitian-unitary-normal.ipynb) and
> [§3.6](../03-eigenvalues/matrix-functions-exponential.ipynb).

## Theory in brief

### States, and the phase that is not there

A **qubit** is a unit vector $|\psi\rangle \in \mathbb{C}^2$, with the
convention that a global phase is unobservable:

```{math}
:label: eq-qb-state
|\psi\rangle \;\sim\; e^{i\varphi}|\psi\rangle ,
\qquad \lVert\,|\psi\rangle\,\rVert = 1 .
```

Everything measurable is a quadratic form in $|\psi\rangle$, and quadratic
forms cannot see $e^{i\varphi}$. The faithful coordinates are the three
Pauli expectation values —

```{math}
:label: eq-qb-bloch
\mathbf{r} \;=\; \bigl(\langle\sigma_x\rangle, \langle\sigma_y\rangle,
\langle\sigma_z\rangle\bigr),
\qquad \langle\sigma_i\rangle = \langle\psi|\sigma_i|\psi\rangle ,
```

the **Bloch vector** of
[§3.4](../03-eigenvalues/hermitian-unitary-normal.ipynb): unit length for
every pure state, and a bijection between states-up-to-phase and points of
the sphere.

### Rotations, from the exponential

For a unit axis $\mathbf{n} \in \mathbb{R}^3$,
$\mathbf{n}\cdot\boldsymbol{\sigma}$ is Hermitian with eigenvalues $\pm1$,
so its exponential collapses to two terms:

```{math}
:label: eq-qb-su2
R_{\mathbf{n}}(\theta) \;=\;
\exp\!\bigl(-i\tfrac{\theta}{2}\,\mathbf{n}\cdot\boldsymbol{\sigma}\bigr)
\;=\; \cos\tfrac{\theta}{2}\,I \;-\;
i\sin\tfrac{\theta}{2}\;\mathbf{n}\cdot\boldsymbol{\sigma} ,
```

because $(\mathbf{n}\cdot\boldsymbol{\sigma})^2 = I$ — the su(2) version
of Euler's formula. Acting by conjugation on {eq}`eq-qb-bloch`,
$R_{\mathbf{n}}(\theta)$ rotates the Bloch vector by the angle $\theta$
about $\mathbf{n}$ — note the *half*-angle inside the matrix against the
*whole* angle on the sphere. That mismatch is the *double cover*
$\mathrm{SU}(2) \to \mathrm{SO}(3)$: the sphere comes back after $2\pi$,
the state only after $4\pi$, and $R_{\mathbf{n}}(2\pi) = -I$ for every
axis.

### Measurement is a quadratic form

Measuring along $\mathbf{n}$ asks the state for the eigen-decomposition of
$\mathbf{n}\cdot\boldsymbol{\sigma} = P_{+} - P_{-}$ and returns $\pm1$
with the **Born rule**

```{math}
:label: eq-qb-born
p_{\pm} \;=\; \langle\psi|P_{\pm}|\psi\rangle
\;=\; \tfrac12\bigl(1 \pm \mathbf{n}\cdot\mathbf{r}\bigr) ,
```

so the expectation value is the plain dot product
$\mathbf{n}\cdot\mathbf{r}$ — geometry doing probability's bookkeeping.

### The state from its statistics

Conversely the statistics rebuild the state. The rank-one projector onto
$|\psi\rangle$ is

```{math}
:label: eq-qb-tomo
|\psi\rangle\langle\psi| \;=\;
\tfrac12\bigl(I + \mathbf{r}\cdot\boldsymbol{\sigma}\bigr) ,
```

so measuring the three Pauli expectations is **tomography**: it recovers
the state exactly, up to the phase that was never there — and
{eq}`eq-qb-tomo` is the door through which density matrices enter in
[§9.4](measurement-channels-choi.ipynb).

---
## Setup

Data and restated instruments only: the Pauli matrices and the named
gates as explicit constants, and the Bloch-coordinate reader built in
[§3.4](../03-eigenvalues/hermitian-unitary-normal.ipynb). The rotation
family, the measurement machinery and the tomography map are all built
in the exercises, where they are the lesson.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import sympy as sp

from ecp import animate, draw, validate
from ecp import linalg as la
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random state below comes from this seed

EPS = np.finfo(float).eps

# data: the Pauli matrices — the coordinate system of the whole chapter.
SIGMA_X = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
SIGMA_Y = np.array([[0.0, -1.0j], [1.0j, 0.0]])
SIGMA_Z = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=complex)
PAULIS = [SIGMA_X, SIGMA_Y, SIGMA_Z]

# data: the named gates, as the explicit constants every text tabulates.
GATE_H = np.array([[1.0, 1.0], [1.0, -1.0]], dtype=complex) / np.sqrt(2.0)
GATE_S = np.diag([1.0, 1.0j])
GATE_T = np.diag([1.0, np.exp(1.0j * np.pi / 4)])

# data: the six landmark states — the poles of the three Pauli axes.
KET0 = np.array([1.0, 0.0], dtype=complex)
KET1 = np.array([0.0, 1.0], dtype=complex)
KET_PLUS = (KET0 + KET1) / np.sqrt(2.0)
KET_MINUS = (KET0 - KET1) / np.sqrt(2.0)
KET_I = (KET0 + 1.0j * KET1) / np.sqrt(2.0)
KET_MI = (KET0 - 1.0j * KET1) / np.sqrt(2.0)


# built from scratch in §3.4; restated here as an instrument.
def bloch_vector(psi):
    """The Bloch coordinates of Eq. 2: the three Pauli expectation values."""
    return np.array([float(np.real(psi.conj() @ s @ psi)) for s in PAULIS])


# instrument: the seeded state factory the exercises draw from — two
# Gaussian draws and a norm, no physics inside.
def random_state(generator):
    """A Haar-ish random pure state: complex Gaussian, normalised."""
    v = generator.standard_normal(2) + 1.0j * generator.standard_normal(2)
    return v / np.linalg.norm(v)

## Exercise 1: The sphere is faithful

{eq}`eq-qb-bloch` claims three numbers pin down a state up to phase.
This exercise checks the geometry from both ends: landmarks land where
the axes say, phases vanish, and lengths are exactly one.

**Part a)** Compute the Bloch vectors of the six landmark states
$|0\rangle, |1\rangle, |{+}\rangle, |{-}\rangle, |{+}i\rangle,
|{-}i\rangle$ from the Setup, and gate each equal to its pole
($\pm\mathbf{e}_z, \pm\mathbf{e}_x, \pm\mathbf{e}_y$ respectively) to
$5\times10^{-15}$ — the dictionary between kets and geography, checked entry
by entry.

**Part b)** Gate the two structural facts on 200 seeded random states
from `random_state(rng)`: $\lVert\mathbf{r}\rVert = 1$ to $10^{-14}$
(purity as geometry), and the global phase invisible —
$\mathbf{r}(e^{i\varphi}|\psi\rangle) = \mathbf{r}(|\psi\rangle)$ to
$5\times10^{-15}$ for a seeded random phase per state, because every entry of
{eq}`eq-qb-bloch` is a quadratic form and $|e^{i\varphi}|^2 = 1$.

**Part c)** Draw the two projections of the sphere ($xz$ and $xy$
discs) with the six landmarks as labelled arrows — the map the rest of
the notebook moves around on.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.below(
    worst_landmark, 5e-15,
    "the six landmark states sit on their poles (Eq. 2)",
    "kets to geography, six entries at a time — the dictionary the whole "
    "chapter reads",
)
validate.below(
    worst_len, 1e-14,
    "every pure state's Bloch vector has length exactly one",
    f"{N_STATES} seeded states: purity is a geometric statement, and the "
    "sphere is not an approximation",
)
validate.below(
    worst_phase, 5e-15,
    "and the global phase is invisible (Eq. 1)",
    "each Bloch entry is a quadratic form, and |e^(i phi)|^2 = 1 — the "
    "equivalence class, measured",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2: Rotations, two routes to one unitary

{eq}`eq-qb-su2` is an identity between a matrix exponential and a
two-term closed form. Both routes are built here — they share no code —
and their agreement is the exercise's first gate; the geometry
($\theta$ on the sphere from $\theta/2$ in the matrix) is its second.

**Part a)** Write `su2_rotation(n, theta)` returning the **closed form**
$\cos(\theta/2)\,I - i\sin(\theta/2)\,\mathbf{n}\cdot\boldsymbol{\sigma}$,
and `su2_spectral(n, theta)` returning
$V e^{-i\theta\Lambda/2} V^{\dagger}$ from `np.linalg.eigh` of
$\mathbf{n}\cdot\boldsymbol{\sigma}$ — the
[§3.6](../03-eigenvalues/matrix-functions-exponential.ipynb) spectral
rule on a Hermitian $2\times2$. Gate the two equal to $10^{-13}$ over
50 seeded axis–angle pairs with $\theta \in [0, 4\pi)$.

**Write these yourself** — the pair is this chapter's engine, and every
gate in every later notebook of the chapter is one of these matrices.

**Part b)** Gate membership in $\mathrm{SU}(2)$: unitarity to
$100\,\varepsilon$ and $\det R = 1$ to $10^{-13}$ over the same pairs
(the determinant is $e^{-i\theta\operatorname{tr}(\mathbf{n}\cdot
\boldsymbol{\sigma})/2} = e^{0}$ — traceless generators land the
rotation in the *special* unitary group).

**Part c)** Gate the half-angle geometry: for 50 seeded (axis, angle,
state) triples, the Bloch vector of $R_{\mathbf{n}}(\theta)|\psi\rangle$
equals the Rodrigues rotation of $\mathbf{r}$ by the **full** angle
$\theta$ about $\mathbf{n}$, to $10^{-12}$ — the matrix turns half as
fast as the sphere, which is the double cover announcing itself.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.below(
    worst_routes, 1e-13,
    "the closed form and the spectral exponential agree (Eq. 3)",
    f"{N_PAIRS} axis-angle pairs across two turns: (n.sigma)^2 = I makes "
    "the series fold into two terms, and 3.6's route confirms it without "
    "sharing a line of code",
)
validate.check(
    worst_unitary < 100 * EPS and worst_det < 1e-13,
    "every rotation lands in SU(2)",
    f"unitarity {worst_unitary:.1e}, |det - 1| {worst_det:.1e}: traceless "
    "generators exponentiate to determinant one",
)
validate.below(
    worst_geo, 1e-12,
    "and the sphere turns by the FULL angle while the matrix turns by half",
    "the Bloch image of R_n(theta)|psi> equals the Rodrigues rotation by "
    "theta — the half-angle bookkeeping that becomes Exercise 4's minus "
    "sign",
)

## Exercise 3: The gate zoo, and one exact eighth root

The named gates of the Setup are specific points of the rotation family,
and their celebrated algebra is a set of $2\times2$ identities cheap to
gate at rounding. One of them is *exactly* true and floating point can
only nearly say so — the chapter's exact-versus-float thread, opened.

**Part a)** Gate the zoo at $5\times10^{-15}$ (entrywise, absolute): $H^2 = I$,
$S^2 = Z$, $T^2 = S$, $HZH = X$, $HXH = Z$, and $\sigma_x\sigma_y =
i\sigma_z$ — the conjugation identities that make $H$ the basis-changer
between the $z$ and $x$ measurement bases.

**Part b)** The eighth root: gate `sympy`'s verdict that $T^8 = I$
**exactly** (build $T$ symbolically with `sp.exp(sp.I*sp.pi/4)` and
simplify $T^8 - I$ to the zero matrix), and *report* the float route's
drift $\lVert T^8 - I\rVert_{\max}$ — measured at a few $\varepsilon$,
gated below $100\,\varepsilon$. Same theorem, two arithmetics, and the
course's standing question — determined by the mathematics or by the
machine? — answered by giving each its own check.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.below(
    worst_zoo, 5e-15,
    "the gate zoo's algebra holds at rounding",
    "six identities, one worst entry — H conjugates Z into X, which is "
    "why one Hadamard changes the measurement basis",
)
validate.check(
    t8_exact and t8_drift < 100 * EPS,
    "T^8 = I exactly in symbols, and to a few eps in floats",
    f"e^(i pi/4) is an eighth root of unity — SymPy says so exactly; "
    f"float64 drifts by {t8_drift:.1e} and is gated only below 100 eps, "
    "because WHICH few-eps value appears belongs to the machine",
)

## Exercise 4: The double cover, measured

Exercise 2 showed the sphere turning at twice the matrix's rate. Paid
forward a full turn, that bookkeeping becomes the subject's most famous
minus sign: $R_{\mathbf{n}}(2\pi) = -I$. Nothing on the sphere moves,
every probability returns to its start — and the sign is still
physically there, readable whenever the rotated state *interferes* with
an unrotated copy.

**Part a)** Gate the sign: for five seeded axes,
$R_{\mathbf{n}}(2\pi) = -I$ and $R_{\mathbf{n}}(4\pi) = +I$, each to
$10^{-13}$ — one turn negates, two restore.

**Part b)** Gate the composition law on a single axis:
$R_{\mathbf{n}}(\alpha)R_{\mathbf{n}}(\beta) =
R_{\mathbf{n}}(\alpha{+}\beta)$ to $10^{-13}$ over 50 seeded pairs —
the one-parameter groups inside $\mathrm{SU}(2)$.

**Part c)** Read the sign by interference: the overlap
$\langle\psi|R_{\mathbf{n}}(\theta)|\psi\rangle$ for $|\psi\rangle$
an eigenstate of $\mathbf{n}\cdot\boldsymbol{\sigma}$ equals
$e^{\mp i\theta/2}$, so its real part traces $\cos(\theta/2)$ — gate the
curve against the cosine to $10^{-13}$ on a 400-point sweep to $4\pi$,
and gate the value $-1$ at exactly $\theta = 2\pi$. Draw it: the
interference fringe that closes after **two** turns.

**Part d)** Animate the precession that Part c's fringe rides on: a
state tilted $60°$ from the pole, evolved by $R_z(\theta)$, its Bloch
vector circling the equator-parallel at constant height while the
overlap's phase winds at half its rate.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.below(
    worst_cover, 1e-13,
    "one full turn is -I, two full turns are +I (Eq. 3 at theta = 2 pi)",
    "the double cover as two matrix identities — nothing observable moved, "
    "and the state still remembers the turn",
)
validate.below(
    worst_comp, 1e-13,
    "rotations about one axis compose by adding angles",
    "50 seeded pairs: the one-parameter subgroup, closed under its own law",
)
validate.check(
    fringe_gap < 1e-13 and abs(at_2pi.real + 1.0) < 1e-13,
    "and interference reads the hidden sign: the fringe is cos(theta/2)",
    f"400 sweep points at {fringe_gap:.1e}, with the overlap exactly -1 "
    "at one full turn — the minus sign is measurable the moment two "
    "histories meet",
)

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


## Exercise 5: Born's rule is a quadratic form

{eq}`eq-qb-born` prices every measurement with a dot product. This
exercise gates the identity, then lets the seeded generator play
experimentalist and checks that finite-shot frequencies land where the
rule says — with the statistical band stated, per the course's rules
for seeded statistics.

**Part a)** Write `measure_probs(psi, n)` returning $(p_+, p_-)$ from
the projectors of $\mathbf{n}\cdot\boldsymbol{\sigma}$ (via
`np.linalg.eigh`), and gate the identity against
$\tfrac12(1 \pm \mathbf{n}\cdot\mathbf{r})$ to $10^{-13}$ over 100
seeded (state, axis) pairs, with $p_+ + p_- = 1$ to $10^{-14}$ —
probability as geometry, both directions.

**Part b)** Gate the expectation identity
$\langle\psi|\,\mathbf{n}\cdot\boldsymbol{\sigma}\,|\psi\rangle =
\mathbf{n}\cdot\mathbf{r}$ to $10^{-13}$ on the same pairs — the
quadratic form and the dot product are the same number.

**Part c)** Simulate: for the tilted state of Exercise 4 measured along
$\mathbf{e}_z$ ($p_+ = \cos^2 30° = 0.75$ exactly), draw $N = 4000$
seeded shots and gate the frequency inside
$p_+ \pm 4\sqrt{p_+(1-p_+)/N} = 0.75 \pm 0.027$ — four standard
errors, stated in advance. Draw the frequency converging into its band.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    worst_born < 1e-13 and worst_sum < 1e-14,
    "the Born rule is the dot-product formula (Eq. 4)",
    f"100 seeded state-axis pairs: projector route vs (1 +- n.r)/2 at "
    f"{worst_born:.1e}, completeness at {worst_sum:.1e}",
)
validate.below(
    worst_expect, 1e-13,
    "and the expectation value is n . r, literally",
    "the quadratic form and the dot product are one number — geometry "
    "doing probability's bookkeeping",
)
validate.below(
    abs(freq_running[-1] - P_TRUE), band_born,
    "4000 seeded shots land inside the stated four-sigma band",
    f"frequency {freq_running[-1]:.4f} against 0.75 exactly (cos^2 30°), "
    f"band +-{band_born:.4f}: the statistics gated generously, the "
    "probability gated exactly",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6: Tomography: the state from its statistics

{eq}`eq-qb-tomo` closes the loop: the three numbers the sphere plots
are enough to rebuild the state. This is one-qubit **state
tomography**, and it is also the door to
[§9.4](measurement-channels-choi.ipynb) — the object
$\tfrac12(I + \mathbf{r}\cdot\boldsymbol{\sigma})$ is the state's
*density matrix*, meeting the reader here as a reconstruction.

**Part a)** Write `state_from_bloch(r)` assembling $\rho =
\tfrac12(I + \mathbf{r}\cdot\boldsymbol{\sigma})$ and returning the
unit eigenvector of its largest eigenvalue (`np.linalg.eigh`). For 100
seeded states, gate the reconstruction up to phase:
$|\langle\hat\psi|\psi\rangle| = 1$ to $10^{-12}$ — the modulus, since
{eq}`eq-qb-state`'s phase was never in the data.

**Part b)** Gate the reconstruction's structure: $\rho$ Hermitian to
$10^{-15}$, trace one to $10^{-15}$, eigenvalues $(1, 0)$ to $10^{-14}$
— a *pure* state's density matrix is a rank-one projector, and its
spectrum says so before any eigenvector is read.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.below(
    worst_overlap, 1e-12,
    "three expectation values rebuild the state, up to phase (Eq. 5)",
    "100 seeded states, overlap modulus 1: tomography is an eigenvector "
    "read off a 2x2 reconstruction, and the phase it cannot recover is "
    "the phase Exercise 1 showed was never observable",
)
validate.check(
    worst_herm < 1e-15 and worst_trace < 1e-15 and worst_spec < 1e-14,
    "and the reconstructed density matrix is a rank-one projector",
    f"Hermitian, unit trace, spectrum (1, 0) at rounding — purity as a "
    "spectral statement, which is exactly where 9.4 picks up",
)

---
## Notebook summary

**The sphere is exact bookkeeping.** Six landmark kets landed on their
poles at $10^{-16}$-scale; 200 seeded states had $\lVert\mathbf{r}\rVert
= 1$ to $10^{-15}$ and phase-invisibility to $10^{-16}$-scale — the
Bloch map is a bijection of states-up-to-phase onto the sphere, measured.

**One rotation, two derivations, half the angle.** The closed su(2) form
and [§3.6](../03-eigenvalues/matrix-functions-exponential.ipynb)'s
spectral route agreed to $2\times10^{-15}$ across two full turns of 50
seeded axes; all landed in $\mathrm{SU}(2)$ at rounding; and the Bloch
image rotated by the **full** angle (Rodrigues, $10^{-16}$-scale) while
the matrix carried half — the double cover in its ledger form.

**The famous minus sign is a measurement.** $R(2\pi) = -I$ and
$R(4\pi) = I$ to $10^{-16}$-scale; the interference fringe
$\mathrm{Re}\langle0|R_z(\theta)|0\rangle$ sat on $\cos(\theta/2)$ at
$10^{-16}$-scale across 400 points and read exactly $-1$ at one turn.
$T^8 = I$ came out **exactly** in SymPy while float64 drifted by
$9\times10^{-16}$ — both gated, each in its own arithmetic.

**Probability is geometry.** Born probabilities matched
$\tfrac12(1 \pm \mathbf{n}\cdot\mathbf{r})$ at $10^{-16}$-scale over
100 state–axis pairs, expectations equalled $\mathbf{n}\cdot\mathbf{r}$
literally, and 4000 seeded shots settled at $0.7478$ inside the
pre-stated $0.75 \pm 0.027$ band. Tomography then ran the arrow
backwards: $\rho = \tfrac12(I + \mathbf{r}\cdot\boldsymbol{\sigma})$
rebuilt every state to overlap modulus 1 at $10^{-13}$-scale, with
spectrum $(1, 0)$ announcing purity.

**Methods introduced.** `su2_rotation` and `su2_spectral` (one identity,
two routes), `rodrigues` as the SO(3) shadow, the gate zoo's conjugation
algebra, SymPy-exact roots of unity beside float drift, `measure_probs`
from projectors, seeded finite-shot bands stated in advance, and
`state_from_bloch` — the density matrix, met as a reconstruction.

## Outlook

- **Two qubits change everything.** The tensor product
  ([§6.4](../06-structure/kronecker-vec-separable.ipynb)'s Kronecker
  product wearing physics clothes) makes the state space grow
  multiplicatively, and single-sphere pictures stop sufficing the moment
  states entangle — [§9.2](tensor-products-entanglement.ipynb) makes the
  failure precise with an SVD.
- **Mixed states fill the ball.** Exercise 6's $\rho$ had spectrum
  $(1,0)$; noise produces every spectrum $(\lambda, 1-\lambda)$, Bloch
  vectors of length below one, and the density-matrix calculus of
  [§9.4](measurement-channels-choi.ipynb).
- **Universality.** $H$ and $T$ alone generate a dense subgroup of
  $\mathrm{SU}(2)$ — the Solovay–Kitaev theorem prices how fast the
  approximation converges, and the T-count of a circuit is the going
  currency of fault-tolerant costing.
- **Berry's phase.** The double cover's sign is the simplest *geometric*
  phase; transport a spin around a closed loop of Hamiltonians and the
  phase picked up is half the solid angle enclosed — the sphere's
  curvature made observable, one step beyond this notebook's fixed axis.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()